---
title: "Part 5: Matplotlib and Its Ecosystem"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/08-matplotlib.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/08-matplotlib.ipynb)

**DS-MLOps Python Foundations**

**Python 3.12+ | Author: Anthony Faustine**

You've just finished cleaning `university_analytics.csv`. The head looks right, the dtypes are correct, the nulls are gone. Then your manager asks: *"What does the score distribution actually look like?"* You could print quartiles. You could sort and read 2,400 rows. Or you could write three lines of matplotlib and see the entire shape in half a second.

Part 4b introduced NumPy arrays as the numerical backbone. This notebook puts those arrays on screen. You will build every standard chart type, compose multi-panel figures, and learn the seaborn shortcut for statistical graphics. Part 6 (`09-lets-plot.ipynb`) introduces a different approach entirely: the grammar of graphics.

> Callout markers: [book cover page](../../index.qmd#callout-guide).

::: {.callout-note collapse="true" icon=false}
## Before You Begin

This notebook follows Parts 1-4. You should have `numpy` and `matplotlib` installed (`pip install matplotlib seaborn`). All code uses the object-oriented `fig, ax = plt.subplots()` pattern -- not the legacy `plt.plot()` state machine.
:::

## Python's Plotting Landscape

You've just finished cleaning `university_analytics.csv`. The head looks right, the dtypes are correct, the nulls are gone. Then your manager asks: *"What does the score distribution actually look like?"* You could print quartiles. You could sort and read through 2,400 rows. Or you could produce one chart that answers the question before the sentence is finished.

That chart needs a library. Python has several, and they make different trade-offs:

| Library | Style | Strengths | Best for |
| --- | --- | --- | --- |
| **Matplotlib** ([matplotlib.org](https://matplotlib.org)) | Imperative (OO API) | Total control, publication quality, runs everywhere | Custom figures, fine-grained layout, saving to PNG/PDF/SVG |
| **Seaborn** ([seaborn.pydata.org](https://seaborn.pydata.org)) | High-level on top of Matplotlib | Statistical plots in one line, beautiful defaults | Distribution plots, pair plots, heatmaps |
| **Lets-Plot** ([lets-plot.org](https://lets-plot.org)) | Declarative, Grammar of Graphics | Expressive, ggplot2-compatible, interactive-ready | Layered charts, Part 6 of this book |
| **Plotly** ([plotly.com/python](https://plotly.com/python)) | Declarative, interactive | Hover, zoom, dash integration | Dashboards, interactive reports |
| **Bokeh** ([bokeh.org](https://docs.bokeh.org)) | Declarative, interactive | Streaming, large data | Real-time visualisations |

This chapter focuses on Matplotlib and Seaborn. Every other library in the list either wraps Matplotlib, exports to it, or assumes you understand it. Matplotlib is the bedrock: learn it once and every other plotting library becomes a set of shortcuts on top of something you already know.

### Already in your environment

Both libraries are in `pyproject.toml`. For a standalone project:

```bash
uv add matplotlib seaborn
```

::: {.callout-note collapse="true" icon=false}
## Learning Objectives

By the end of Part 5 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Explain the Figure/Axes object model and why it matters | Sec. 1 |
| 2 | Build line, scatter, bar, and histogram charts with the object-oriented API | Sec. 2 |
| 3 | Lay out and compare multiple Axes in one Figure | Sec. 3 |
| 4 | Save a figure at the right resolution and format for its destination | Sec. 3 |
| 5 | Use seaborn for one-line statistical graphics, then keep customising with matplotlib | Sec. 4 |
:::


## 1. Why Visualise? The Figure and Axes

A table of a thousand exam scores tells you nothing at a glance. A histogram of the same thousand scores tells you the shape of the distribution in about half a second. That is the entire case for visualisation: it trades a small amount of precision for a large amount of immediate understanding, which is exactly what you want before you have decided which question to ask next.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
exam_scores = rng.normal(loc=72, scale=12, size=1000).clip(0, 100)

print(f"mean   : {exam_scores.mean():.1f}")
print(f"median : {np.median(exam_scores):.1f}")
print(f"std    : {exam_scores.std():.1f}")
# The numbers alone do not tell you whether the distribution is symmetric,
# has a long tail, or is bimodal. A histogram answers that in one look.

matplotlib has two APIs for building the same chart. The older one, `pyplot` (`plt.plot(...)`), is a state machine: it always draws onto "whichever figure was most recently touched," which is fine for a single quick chart and confusing the moment you need two charts side by side. The **object-oriented API** is explicit instead: you ask for a `Figure` (the whole canvas) and one or more `Axes` (an individual plot inside it), then call methods directly on the `Axes` you want to draw on.

In [ ]:
import matplotlib.pyplot as plt

# The object-oriented pattern you will use for almost every chart in this
# notebook: ask for a Figure and an Axes, then call methods on the Axes.
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(exam_scores, bins=20, color="#4477AA", edgecolor="white")
ax.set_xlabel("Exam score")
ax.set_ylabel("Number of students")
ax.set_title("Exam score distribution");

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Figure vs. Axes</span><br><br>
A <code>Figure</code> is the whole canvas: the window or page a chart is drawn on, and the thing you save to a file. An <code>Axes</code> is one plot inside that canvas, with its own x-axis, y-axis, title, and data. <code>fig, ax = plt.subplots()</code> gives you one of each. Every method that actually draws data (<code>.plot()</code>, <code>.scatter()</code>, <code>.bar()</code>, <code>.hist()</code>) lives on the <code>Axes</code>, not the <code>Figure</code>.
</div>

The dashed boundary below is the one thing in this diagram that has no real line of code behind it: it's just there to make the Figure's own edge visible, since otherwise it's easy to forget it exists at all.

![matplotlib object model: the Figure is the outer canvas containing one or more Axes. The Axes holds the plot area with title, axis labels, ticks, data, and optional legend.](figs/figure-axes-anatomy.svg){fig-alt="Outer blue-bordered rectangle labeled Figure containing a gray-bordered rectangle labeled Axes with a bar chart inside. Callout annotations point to title, xlabel, ylabel, ticks, and legend."}

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Mixing <code>plt.plot()</code> and <code>ax.plot()</code></span><br><br>
<code>plt.title("x")</code> sets the title of whichever Axes <code>pyplot</code> thinks is "current", which silently changes after you create a new subplot. The moment you have more than one Axes, calling <code>plt.xlabel()</code> instead of <code>ax.set_xlabel()</code> is a common way to label the wrong chart. Once you have an <code>ax</code> object, call methods on it directly and skip <code>plt.*</code> entirely.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Your First Figure/Axes Chart</span><br><br>
<b>Goal:</b> Build a histogram using the object-oriented pattern.<br><br>1. Create 500 normally distributed exam scores: <code>scores = np.random.default_rng(42).normal(72, 12, 500).clip(0, 100)</code>.<br>2. Call <code>fig, ax = plt.subplots(figsize=(7, 4))</code>.<br>3. Plot a histogram with 20 bins: <code>ax.hist(scores, bins=20, color='#0369A1', edgecolor='white')</code>.<br>4. Add a title and x/y labels with <code>ax.set_title()</code>, <code>ax.set_xlabel()</code>, <code>ax.set_ylabel()</code>.<br>5. Add a vertical line at the mean: <code>ax.axvline(scores.mean(), color='#DC2626', lw=2, label='mean')</code>.<br><br><b>Expected:</b> a labelled histogram with a red mean line.
</div>

## 2. Core Chart Types

Four chart types cover most of what you will plot day to day. Each answers a different question, and picking the wrong one is the fastest way to make a chart that looks fine but says nothing useful.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: Match chart type to the question, not the data</span><br><br>
Each chart type answers one question well and others poorly:<br><b>Line</b>: how does a value change over time or sequence?<br><b>Scatter</b>: is there a relationship between two continuous variables?<br><b>Bar</b>: how do discrete categories compare?<br><b>Histogram</b>: what is the distribution of a single variable?<br>Picking the wrong type is the fastest way to make a chart that looks fine but says nothing.
</div>

In [ ]:
# Build the running dataset: exam results across three courses and two
# semesters for the university analytics platform.
rng = np.random.default_rng(42)

courses = np.array(["Machine Learning", "Data Structures", "Statistics"])
semesters = np.array(["Fall 2024", "Spring 2025"])

n_per_group = 60
course_col = np.repeat(courses, n_per_group * len(semesters))
semester_col = np.tile(np.repeat(semesters, n_per_group), len(courses))

# Each course has a slightly different difficulty and improves slightly
# from Fall to Spring, to give the line chart in this section something
# real to show.
course_base = {"Machine Learning": 68, "Data Structures": 74, "Statistics": 71}
semester_bump = {"Fall 2024": 0, "Spring 2025": 4}

exam_score = np.array(
    [rng.normal(course_base[c] + semester_bump[s], 10) for c, s in zip(course_col, semester_col, strict=True)]
).clip(0, 100)
study_hours = rng.uniform(0, 25, size=len(course_col))
attendance_pct = rng.uniform(50, 100, size=len(course_col))

print(f"rows: {len(course_col)}")
print(f"courses: {courses}")

**Line chart**, for a trend across an ordered sequence. Here, average score per course from Fall to Spring:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))

for course in courses:
    course_mask = course_col == course
    means = [exam_score[course_mask & (semester_col == s)].mean() for s in semesters]
    ax.plot(semesters, means, marker="o", label=course)

ax.set_ylabel("Average exam score")
ax.set_title("Average score by course, Fall to Spring")
ax.legend();

**Scatter plot**, for the relationship between two continuous variables. Here, study hours against exam score:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(study_hours, exam_score, alpha=0.4, color="#4477AA")
ax.set_xlabel("Study hours")
ax.set_ylabel("Exam score")
ax.set_title("Study hours vs. exam score");

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Add an infinite reference line with <code>ax.axline()</code></span><br><br>
<code>ax.axline(xy1, slope=m)</code> draws an infinite line through point <code>xy1</code> at gradient <code>m</code>. Unlike <code>ax.plot()</code>, which clips to the data you provide, <code>axline</code> always extends to the full axis range even when the axes limits change later. Use it to add a <code>y = x</code> diagonal (slope 1 through the origin) on any scatter of two variables that should ideally be equal.
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(study_hours, exam_score, alpha=0.2, s=8, color="#38BDF8")
ax.axline((0, 0), slope=4, color="#DC2626", linewidth=1.5, linestyle="--", label="reference slope")
ax.set_xlabel("Study Hours")
ax.set_ylabel("Exam Score")
ax.set_title("Are high study hours linked to higher scores?")
ax.legend()
plt.tight_layout()
plt.show()

**Bar chart**, for comparing a single number across categories. Here, average score per course:

In [ ]:
course_means = [exam_score[course_col == c].mean() for c in courses]

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(courses, course_means, color=["#4477AA", "#EE6677", "#228833"])
ax.set_ylabel("Average exam score")
ax.set_title("Average score by course")
ax.tick_params(axis="x", labelrotation=15);

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>ax.bar_label()</code> to annotate bar values automatically</span><br><br>
Adding the numeric value above each bar used to mean a manual loop calling <code>ax.text()</code> for each rectangle. Since matplotlib 3.4, <code>ax.bar_label(container)</code> does it in one line. <code>ax.bar()</code> returns a <code>BarContainer</code>; pass it to <code>bar_label</code> and optionally format the numbers with <code>fmt</code>:

<pre style='background:#F4F5F6;padding:10px;border-radius:4px;font-size:0.9em'>bars = ax.bar(courses, course_means, color=["#4477AA", "#EE6677", "#228833"])
ax.bar_label(bars, fmt="{:.1f}", padding=3)</pre>

<code>fmt</code> accepts a format string (applied to each value) or a <code>labels</code> keyword with an explicit list. <code>padding</code> pushes the text a few points above the bar top.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Attendance Histogram</span><br><br>

<b>Goal:</b> Plot a histogram of <code>attendance_pct</code> with 15 bins, label both axes, and give it a title. Use the object-oriented pattern: <code>fig, ax = plt.subplots()</code>, then call methods on <code>ax</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(attendance_pct, bins=15, ...)
# expect a roughly uniform spread between 50 and 100, since
# attendance_pct was generated with rng.uniform(50, 100, ...)</pre>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
# TODO: plot the histogram, then set xlabel, ylabel, and title
...

fig

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: A trailing bare <code>fig</code> displays the figure in Jupyter</span><br><br>
Jupyter displays the last expression in a cell automatically, the same way it prints a bare <code>x</code> on its own line. Ending a plotting cell with <code>fig</code> (or letting <code>ax.hist(...)</code> be the last call) shows the chart without an explicit <code>plt.show()</code>, which you only need outside a notebook.
</div>

## 3. Multiple Axes and Saving Figures

Real analysis rarely stops at one chart. `plt.subplots(nrows, ncols)` returns a whole grid of `Axes` at once, as a NumPy array, so you can loop over it the same way you looped over any other array in Part 4.

<div style='background:#EFF6FF;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-key-fill"></i> Key Concept: subplots returns a NumPy array of Axes -- loop over it with .flat</span><br><br>
<code>fig, axes = plt.subplots(2, 3)</code> gives you a <code>(2, 3)</code> NumPy array of <code>Axes</code> objects. Use <code>axes.flat</code> to iterate over all six in order, regardless of grid shape. <code>sharey=True</code> links the y-axis scale across all panels so comparisons stay honest.
</div>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)

for ax, course in zip(axes.flat, courses, strict=True):
    course_scores = exam_score[course_col == course]
    ax.hist(course_scores, bins=15, color="#4477AA", edgecolor="white")
    ax.set_title(course, fontsize=10)
    ax.set_xlabel("Exam score")

axes[0].set_ylabel("Number of students")
fig.suptitle("Score distribution per course");

`axes.flat` works regardless of the grid shape: a `2x2` grid of `Axes` is a 2D array, but `.flat` always gives you a flat iterator over all of them, in row-major order. `sharey=True` forces every Axes in the grid to use the same y-axis range, which is what makes the three histograms above honestly comparable instead of each rescaled to its own data.

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Comparing histograms with different y-axis scales</span><br><br>
Without <code>sharey=True</code>, matplotlib autoscales each Axes to its own data. Three histograms that look like they have the same number of students can actually have wildly different counts, because each y-axis silently uses a different scale. Whenever you put similar charts side by side for comparison, force a shared scale.
</div>

Saving a figure has two choices that matter: resolution (`dpi`, dots per inch) and file format. A raster format (PNG) at low `dpi` looks blurry when scaled up; a vector format (SVG or PDF) stays sharp at any size because it stores shapes, not pixels. Call `fig.tight_layout()` before saving to close any gaps between subplots and prevent axis labels from being clipped:

In [ ]:
from pathlib import Path

output_dir = Path("tmp_plots")
output_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(exam_scores, bins=20, color="#4477AA", edgecolor="white")
ax.set_title("Exam score distribution")
fig.tight_layout()

# PNG: fine for slides and notebooks, blurry if you zoom in or print large
fig.savefig(output_dir / "scores.png", dpi=150, bbox_inches="tight")
# SVG: vector format, stays crisp at any size, the right choice for reports
fig.savefig(output_dir / "scores.svg", bbox_inches="tight")

print(sorted(p.name for p in output_dir.iterdir()))

import shutil

shutil.rmtree(output_dir)

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Default to vector formats for anything that gets printed or zoomed</span><br><br>
PNG and JPEG store a fixed grid of pixels: stretch them and they blur. SVG and PDF store the actual shapes and text, so they render sharp at any zoom level or print size. Save PNG for quick previews and web thumbnails; save SVG or PDF for anything that ends up in a report, slide deck, or paper.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Three-Panel Score Comparison</span><br><br>
<b>Goal:</b> Create a 1x3 grid comparing scores across three groups.<br><br>1. Create three arrays: <code>maths = np.array([78,82,91,65,88,74,95,61,83,79])</code>, <code>stats = np.array([72,88,65,91,77,83,69,94,71,86])</code>, <code>prog  = np.array([85,79,92,68,88,76,93,71,84,90])</code>.<br>2. Call <code>fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharey=True)</code>.<br>3. Loop over <code>zip(axes.flat, ['Maths','Stats','Programming'], [maths, stats, prog])</code> and plot a histogram on each axis.<br>4. Set a title on each axis and a shared y-label on the leftmost axis only.<br><br><b>Expected:</b> three side-by-side histograms sharing the same y-scale.
</div>

## 4. Seaborn: Statistical Graphics in One Line

Seaborn is built directly on matplotlib. It does not replace anything from Sections 1-3: it adds a layer of functions that know how to take a whole DataFrame, split it by a category, color each group, and draw a legend, all in a single call. Reaching for seaborn first whenever your chart needs grouping or a statistical summary saves a genuine amount of code.

Seaborn expects **tidy data**: one row per observation, one column per variable. A full pandas introduction comes later in the data analysis tutorials, but building a DataFrame from arrays you already have is one line:

In [ ]:
import pandas as pd

results = pd.DataFrame(
    {
        "course": course_col,
        "semester": semester_col,
        "exam_score": exam_score,
        "study_hours": study_hours,
        "attendance_pct": attendance_pct,
    }
)
results.head()

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.histplot(data=results, x="exam_score", hue="course", kde=True, ax=ax)
ax.set_title("Exam score distribution by course");

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Seaborn returns a matplotlib Axes</span><br><br>
<code>sns.histplot(..., ax=ax)</code> draws onto the <code>Axes</code> you pass it and returns that same <code>Axes</code>. Nothing from Sections 1-3 is wasted: every <code>ax.set_title()</code>, <code>ax.set_xlabel()</code>, or <code>fig.savefig()</code> you already know still works on a seaborn chart. Seaborn only replaces the part where you would otherwise have looped over groups and called <code>ax.hist()</code> once per group yourself.
</div>

`hue` is seaborn's primary grouping parameter: pass a column name and seaborn splits the data by that column, assigns each group a colour from its default palette, and draws a legend automatically. `palette` overrides those colours, accepting a named seaborn palette (`"tab10"`, `"Set2"`) or a list of hex codes. `style` (available in `sns.lineplot` and `sns.scatterplot`) adds a second visual channel by varying the marker shape or line style per group, which helps when a chart may be viewed in greyscale.

`sns.boxplot` summarises a whole distribution (median, quartiles, outliers) per category in one call, the kind of comparison that would take a loop and several `ax.hist()` calls in raw matplotlib:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.boxplot(data=results, x="course", y="exam_score", hue="semester", ax=ax)
ax.set_title("Exam score by course and semester");

`sns.violinplot()` shows the full shape of the distribution on both sides of a central axis, not just the five-number summary a boxplot gives. Use it when you suspect a distribution is skewed or has more than one peak in a way the box would hide:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.violinplot(data=results, x="course", y="exam_score", hue="semester", ax=ax)
ax.set_title("Exam score distribution by course and semester");

`sns.heatmap` is the standard way to visualise a correlation matrix. Pass it `results[numeric_cols].corr()`, a small DataFrame seaborn happily turns into a colour grid with the actual numbers annotated:

In [ ]:
numeric_cols = ["exam_score", "study_hours", "attendance_pct"]
corr = results[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(4, 3.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature correlation");

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 4 - Compare Study Habits Across Courses</span><br><br>

<b>Goal:</b> Use <code>sns.boxplot</code> to compare <code>study_hours</code> across the three courses (x-axis: <code>course</code>, y-axis: <code>study_hours</code>), then add a title with <code>ax.set_title()</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>fig, ax = plt.subplots(figsize=(6, 3.5))
sns.boxplot(data=results, x="course", y="study_hours", ax=ax)
ax.set_title(...)</pre>
<b>Hint:</b> This is almost identical to the boxplot above, just with a different y-axis and no <code>hue</code>.
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
# TODO: boxplot of study_hours by course, plus a title
...

fig

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Exploratory charts and presentation charts have different goals</span><br><br>
Seaborn's defaults are optimised for <b>exploratory</b> use: fast, readable charts that help you understand the data before you have decided what question to ask. A <b>presentation</b> chart, one headed for a report or a slide deck, needs deliberate title wording, axis labels in the reader's language, and a colour palette that matches the project's house style. Part 7 covers that transition.
</div>

## 5. Flexible Layouts with subplot_mosaic

The `subplots(rows, cols)` grid forces every panel to the same size. `fig.subplot_mosaic()` lets you describe a layout by name: a wide chart on top and two narrow ones below, a tall panel beside a grid of small ones. You draw the layout as a list of strings where each letter names a panel.

In [ ]:
mosaic = [["hist", "hist"], ["scatter", "box"]]
fig, axes = plt.subplot_mosaic(mosaic, figsize=(10, 6), constrained_layout=True)

# Wide histogram on top
axes["hist"].hist(results["exam_score"].dropna(), bins=20, color="#38BDF8", edgecolor="white")
axes["hist"].set_title("Exam Score Distribution")

# Scatter bottom-left
axes["scatter"].scatter(results["study_hours"], results["exam_score"], alpha=0.3, color="#7C3AED", s=8)
axes["scatter"].set_xlabel("Study Hours")
axes["scatter"].set_ylabel("Exam Score")

# Boxplot bottom-right
box_data = [results.loc[results["course"] == c, "exam_score"].dropna() for c in courses]
axes["box"].boxplot(box_data, labels=courses)
axes["box"].set_xticklabels(courses, rotation=45, ha="right")

plt.suptitle("Course Performance Overview", fontweight="bold")
plt.show()

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Mosaic layouts use repeated letters to span cells</span><br><br>
The mosaic <code>[["hist", "hist"], ["scatter", "box"]]</code> means: row 1 has one panel called <code>"hist"</code> spanning two columns; row 2 has <code>"scatter"</code> and <code>"box"</code> side by side. Repeated letters span multiple cells. Reference each panel by name via <code>axes["hist"]</code>. The layout reads like ASCII art, which makes it easy to reason about before you see the output.
</div>

## Capstone: A Three-Panel Course Report

Combine everything from this notebook into one Figure with three Axes side by side: a histogram of scores for one course, a scatter of study hours against score for the same course, and a bar chart comparing average scores across all three courses. This is the shape of report you would actually hand to a department head.

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Three-Panel Report</span><br><br>

<b>Goal:</b> Build a <code>(1, 3)</code> grid of Axes:
<ol>
<li>Axes 0: histogram of <code>exam_score</code> for <code>"Machine Learning"</code> only</li>
<li>Axes 1: scatter of <code>study_hours</code> vs. <code>exam_score</code>, same course only</li>
<li>Axes 2: bar chart of average <code>exam_score</code> per course (all courses)</li>
</ol>
Give the Figure an overall title with <code>fig.suptitle()</code> and each Axes its own <code>ax.set_title()</code>. <b>Hint:</b> Filter with <code>ml_mask = results["course"] == "Machine Learning"</code>, then index <code>results[ml_mask]</code> for the first two panels.
</div>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

ml_mask = results["course"] == "Machine Learning"
ml_results = results[ml_mask]

# TODO panel 0: histogram of ml_results["exam_score"]
# TODO panel 1: scatter of ml_results["study_hours"] vs ml_results["exam_score"]
# TODO panel 2: bar chart of average exam_score per course (all courses)
...

fig.suptitle("Course Report: Machine Learning")
fig

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: seaborn 0.12+ ships a declarative objects API alongside the classic functions</span><br><br>
Seaborn 0.12 introduced <code>seaborn.objects</code> (<code>so.Plot()</code>), a fully composable layer built on the same grammar-of-graphics ideas as Part 6's Lets-Plot. It's worth knowing even if you do not switch immediately:

<pre style='background:#F4F5F6;padding:10px;border-radius:4px;font-size:0.9em'>import seaborn.objects as so

(
    so.Plot(results, x="study_hours", y="exam_score", color="course")
    .add(so.Dot(alpha=0.4))
    .label(title="Study hours vs. exam score")
)</pre>

<code>so.Plot()</code> is lazy (nothing renders until you call <code>.show()</code> or display it), composable (chain <code>.add()</code> calls to layer marks), and consistent with the Lets-Plot mental model from Part 6. For exploratory work the classic <code>sns.*</code> functions are still faster to type; <code>so.Plot()</code> pays off when a chart needs several layers or custom marks.
</div>

## Further Reading

| Resource | Why it matters |
|---|---|
| Hunter, J.D. (2007). [Matplotlib: A 2D graphics environment](https://doi.org/10.1109/MCSE.2007.55). *Computing in Science & Engineering* 9(3), 90–95. | The original paper; understanding the Figure/Axes object model it describes makes every API decision predictable |
| VanderPlas, J. (2016). *Python Data Science Handbook*, Ch. 4. O'Reilly. | Free online: covers customisation, subplots, and the stateful vs object-oriented API |
| [Matplotlib documentation: Axes API](https://matplotlib.org/stable/api/axes_api.html) | Every method you will use in practice lives here; use `Ctrl+F` instead of guessing |
| Wilke, C.O. (2019). *Fundamentals of Data Visualization*. O'Reilly. | Free at [clauswilke.com/dataviz](https://clauswilke.com/dataviz): the chapter on figure design explains *why* certain defaults are problematic |


## Summary

| Concept | Key rule |
|---|---|
| Figure vs. Axes | Figure is the canvas, Axes is one plot; draw by calling methods on `ax`, not `plt` |
| `plt.subplots()` | Returns `(fig, ax)` for one plot, `(fig, axes)` for a grid |
| Line chart | Trend across an ordered sequence: `ax.plot()` |
| Scatter plot | Relationship between two continuous variables: `ax.scatter()` |
| Bar chart | Comparing one number across categories: `ax.bar()` |
| Histogram | Shape of one variable's distribution: `ax.hist()` |
| `axes.flat` | Flat iterator over any subplot grid, regardless of its shape |
| `sharey=True` | Forces a fair visual comparison across subplots |
| `fig.tight_layout()` | Closes gaps and prevents clipped labels before saving |
| Saving figures | PNG for previews, SVG/PDF for anything printed or zoomed; pass `bbox_inches="tight"` |
| Seaborn | One-line statistical graphics on tidy DataFrames; returns a matplotlib `Axes` |
| `hue` | Splits data by a column, assigns colours automatically, and draws a legend |
| `sns.violinplot` | Shows the full distribution shape per group, where a boxplot shows only a five-number summary |

**Next:** `09-lets-plot.ipynb`, introducing the grammar of graphics: the same charts from this notebook, built declaratively instead of imperatively.